In [116]:
import rasterio
import xarray as xr
import glob
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from scipy.ndimage import median_filter
from rasterio.io import MemoryFile
from rasterio.features import shapes
from shapely.geometry import shape
#from skimage.morphology import (
#    binary_opening, binary_closing, binary_erosion, binary_dilation,
#    disk, remove_small_objects, remove_small_holes
#)
#from scipy.ndimage import binary_fill_holes

import skimage as ski
import numpy as np

In [117]:
year = 2017

### 1) Define input and output folder

In [118]:
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results")
output_folder.mkdir(exist_ok=True)

input_folder_s2 = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics")
input_folder_s1 = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/" + str(year))

### 2) Find all files and load them

In [119]:
def find_files(input_folder, pattern):
    #Find files in the input folder matching the given pattern
    return sorted(glob.glob(str(input_folder / pattern)))

In [120]:
# find corresponding coherence file
#s1_coherence_files = find_files(input_folder_s1 , "sar_coherence_composite_all_swaths_thresh_0.3_masked.tiff")
#print(f"Found {len(s1_coherence_files)} S1 coherence files.")

# find acending and descending coherence files
s1_coherence_files_asc = find_files(input_folder_s1 , "masked_ascending_0.5.tif")
print(f"Found {len(s1_coherence_files_asc)} S1 coherence files (ascending).")

s1_coherence_files_desc = find_files(input_folder_s1 , "masked_descending_0.5.tif")
print(f"Found {len(s1_coherence_files_desc)} S1 coherence files (descending).")

# find corresponding NDSI file
s2_ndsi_files = find_files(Path(input_folder_s2 / "ndsi" / str(year) ), "ndsi_composite_" + str(year) + "_0.4.tiff")
print(f"Found {len(s2_ndsi_files)} S2 NDSI files.")

# find corresponding ratio file
s2_ratio_files = find_files(Path(input_folder_s2 / "ratio" / str(year) ), "ratio_composite_" + str(year) + ".tiff")
print(f"Found {len(s2_ratio_files)} S2 ratio files.")

with rasterio.open(s1_coherence_files_asc[0]) as src:
    print("Loading S1 coherence (ascending):", s1_coherence_files_asc)
    s1_asc = src.read()
    s1_asc_meta = src.meta
    
with rasterio.open(s1_coherence_files_desc[0]) as src:
    print("Loading S1 coherence (descending):", s1_coherence_files_desc)
    s1_desc = src.read()
    s1_desc_meta = src.meta
   

with rasterio.open(s2_ndsi_files[0]) as src:
    print("Loading S2 NDSI:", s2_ndsi_files)  
    s2_ndsi = src.read()
    s2_ndsi_meta = src.meta
    

with rasterio.open(s2_ratio_files[0]) as src:
    print("Loading S2 ratio:", s2_ratio_files)  
    s2_ratio = src.read()
    s2_ratio_meta = src.meta
      


Found 1 S1 coherence files (ascending).
Found 1 S1 coherence files (descending).
Found 1 S2 NDSI files.
Found 1 S2 ratio files.
Loading S1 coherence (ascending): ['/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/masked_ascending_0.5.tif']
Loading S1 coherence (descending): ['/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/masked_descending_0.5.tif']
Loading S2 NDSI: ['/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/2017/ndsi_composite_2017_0.4.tiff']
Loading S2 ratio: ['/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/2017/ratio_composite_2017.tiff']


### 3) Align them spatially

In [121]:
def align_to_reference(src_array, src_transform, src_crs, ref_meta, resampling=Resampling.nearest):
    #Resample src_array onto the reference grid defined by ref_meta.
    aligned = np.zeros((ref_meta['height'], ref_meta['width']), dtype=src_array.dtype)
    reproject(
        source=src_array,
        destination=aligned,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=ref_meta['transform'],
        dst_crs=ref_meta['crs'],
        resampling=resampling
    )
    return aligned

In [122]:
ref_meta = s1_asc_meta  

s2_ndsi_aligned = align_to_reference(
    s2_ndsi, s2_ndsi_meta['transform'], s2_ndsi_meta['crs'], ref_meta,
    resampling=Resampling.nearest
)

s2_ratio_aligned = align_to_reference(
    s2_ratio, s2_ratio_meta['transform'], s2_ratio_meta['crs'], ref_meta,
    resampling=Resampling.nearest
)

s1_asc_aligned = align_to_reference(
    s1_asc, s1_asc_meta['transform'], s1_asc_meta['crs'], ref_meta, resampling=Resampling.nearest
)

s1_desc_aligned = align_to_reference(
    s1_desc, s1_desc_meta['transform'], s1_desc_meta['crs'], ref_meta, resampling=Resampling.nearest
)

### 5) Combine files and safe to disk

### 5.1) S1 and S2 classification

In [123]:
s2_ndsi_mask = s2_ndsi_aligned.astype(bool)
s2_ratio_mask = s2_ratio_aligned.astype(bool)
s1_asc = s1_asc_aligned.astype(bool)
s1_desc = s1_desc_aligned.astype(bool)

In [124]:
combined = s2_ndsi_mask | s2_ratio_mask | s1_asc | s1_desc

#filtered_raster = median_filter(combined, size=3)

out_meta = ref_meta.copy()
out_meta.update({
    "count": 1,
    "nodata": 0
})

with rasterio.open(output_folder / "S1_S2" / f"s1_s2_asc_desc_combined_{year}_0.4.tiff", "w", **out_meta) as dest:
    dest.write(combined, 1)

### apply filtering

In [125]:
struct = ski.morphology.disk(5)  

# speckle / isolated false-positive pixels
opened = ski.morphology.opening(combined, struct)

# bridge small gaps 
closed = ski.morphology.closing(opened, struct)

# erosion
#eroded = ski.morphology.erosion(combined, struct)

with rasterio.open(output_folder / "S1_S2" /f"s1_s2_asc_desc_morphFilter_5_0.4_{year}.tiff", "w", **out_meta) as dest:
    dest.write(closed, 1) 

### Convert into polygons

In [126]:
results = (
    {'geometry': shape(geom), 'value': val}
    for geom, val in shapes(closed.astype(np.uint8), transform=ref_meta['transform'])
    if val == 1
)

gdf = gpd.GeoDataFrame.from_records(results)
gdf = gdf.set_geometry('geometry')
gdf = gdf.set_crs(ref_meta['crs'])

# save to disk
output_path = output_folder / "S1_S2" / f"s1_s2_glacier_outlines_{year}_v0.4.shp"
gdf.to_file(output_path)
print(f"Saved {len(gdf)} polygons to {output_path}")

Saved 192 polygons to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/s1_s2_glacier_outlines_2017_v0.4.shp


### 5.2) S2 only

In [127]:
combined = s2_ndsi_mask | s2_ratio_mask

#filtered_raster = median_filter(combined, size=3)

out_meta = ref_meta.copy()
out_meta.update({
    "count": 1,
    "nodata": 0
})

with rasterio.open(output_folder / "S2" / f"s2_combined_{year}_0.4.tiff", "w", **out_meta) as dest:
    dest.write(combined, 1)

### Apply filtering

In [128]:
struct = ski.morphology.disk(5)  

# speckle / isolated false-positive pixels
opened = ski.morphology.opening(combined, struct)

# bridge small gaps 
closed = ski.morphology.closing(opened, struct)

# erosion
#eroded = ski.morphology.erosion(combined, struct)

with rasterio.open(output_folder / "S2" / f"s2_morphFilter_5_0.4_{year}.tiff", "w", **out_meta) as dest:
    dest.write(closed, 1) 

### Convert into Polygons

In [129]:
results = (
    {'geometry': shape(geom), 'value': val}
    for geom, val in shapes(closed.astype(np.uint8), transform=ref_meta['transform'])
    if val == 1
)

gdf = gpd.GeoDataFrame.from_records(results)
gdf = gdf.set_geometry('geometry')
gdf = gdf.set_crs(ref_meta['crs'])

# save to disk
output_path = output_folder / "S2" / f"s2_glacier_outlines_{year}_v0.4.shp"
gdf.to_file(output_path)
print(f"Saved {len(gdf)} polygons to {output_path}")

Saved 152 polygons to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/s2_glacier_outlines_2017_v0.4.shp


### 5.3) S1 only

In [130]:
combined = s1_asc | s1_desc

#filtered_raster = median_filter(combined, size=3)

out_meta = ref_meta.copy()
out_meta.update({
    "count": 1,
    "nodata": 0
})

with rasterio.open(output_folder / "S1"/ f"s1_asc_desc_combined_{year}_0.4.tiff", "w", **out_meta) as dest:
    dest.write(combined, 1)

### Apply filtering

In [131]:
struct = ski.morphology.disk(5)  

# speckle / isolated false-positive pixels
opened = ski.morphology.opening(combined, struct)

# bridge small gaps 
closed = ski.morphology.closing(opened, struct)

# erosion
#eroded = ski.morphology.erosion(combined, struct)

with rasterio.open(output_folder / "S1" / f"s1_morphFilter_5_0.4_{year}.tiff", "w", **out_meta) as dest:
    dest.write(closed, 1) 

### Convert into polygons

In [132]:
results = (
    {'geometry': shape(geom), 'value': val}
    for geom, val in shapes(closed.astype(np.uint8), transform=ref_meta['transform'])
    if val == 1
)

gdf = gpd.GeoDataFrame.from_records(results)
gdf = gdf.set_geometry('geometry')
gdf = gdf.set_crs(ref_meta['crs'])

# save to disk
output_path = output_folder / "S1" / f"s1_glacier_outlines_{year}_v0.4.shp"
gdf.to_file(output_path)
print(f"Saved {len(gdf)} polygons to {output_path}")

Saved 211 polygons to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1/s1_glacier_outlines_2017_v0.4.shp
